# Scenario 07 — Agent-Driven Scheduling

This scenario shows how an **agent key** can drive the on-chain scheduling
side of a task — `register`, `attempt` — on behalf of a master, without the
master ever signing. It mirrors the real-world pattern used by The Order and
similar external schedulers.

**Role:** Client + Agent (acting for Master).

**Important:** This scenario only covers the parts the SDK can drive. The
response submission step (`Nexus.submitResponse`) is intentionally **not**
exposed in the SDK and lives only inside the docker source runtime —
otherwise providers could fabricate responses without doing real work. So
this notebook stops at `Task ATTEMPTED`. To carry the task all the way to
`FINALIZED`, you need an actual provider runtime to produce and submit a
real response payload.

**Prerequisites:**

- `CLIENT_PRIVATE_KEY` — funded client wallet
- `AGENT_PRIVATE_KEY` — funded agent wallet (this is the `signer` for every
  master-side call; the master itself does not sign anything in this notebook)
- `MASTER_ADDR`, `PROVIDER_ADDR` — addresses of an already-paired master and
  provider (pairing is the Provider App's job)
- **Scenario 06 already run** — agent is authorized on `Terminal.setAgent`

**Flow:**

```
Client                      Agent (for Master)              Chain
  │                              │                            │
  ├─ 1. publish_source ─────────────────────────────────────→ Source ACTIVE
  ├─ 2. publish_task ──────────────────────────────────────→ Task NEW
  │                              │                            │
  │                              ├─ 3. register (agent key)─→ provider registered
  │                              └─ 4. attempt   (agent key)─→ Task ATTEMPTED
  │
  │                  (response submission lives in the docker source —
  │                   not part of this notebook)
```


## 0. Setup


In [ ]:
import os
import time

from eth_account import Account
from web3 import Web3

from ogpu import ChainConfig, ChainId
from ogpu.client import (
    DeliveryMethod,
    ImageEnvironments,
    SourceInfo,
    TaskInfo,
    TaskInput,
    publish_source,
    publish_task,
)
from ogpu.protocol import nexus, terminal
from ogpu.types import Environment

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

CLIENT_KEY = os.environ["CLIENT_PRIVATE_KEY"]
AGENT_KEY  = os.environ["AGENT_PRIVATE_KEY"]

CLIENT_ADDR = Account.from_key(CLIENT_KEY).address
AGENT_ADDR  = Account.from_key(AGENT_KEY).address

# Fill these in with your actual paired master + provider addresses
MASTER_ADDR   = os.environ["MASTER_ADDR"]
PROVIDER_ADDR = os.environ["PROVIDER_ADDR"]

print(f"Client  : {CLIENT_ADDR}")
print(f"Agent   : {AGENT_ADDR}")
print(f"Master  : {MASTER_ADDR}")
print(f"Provider: {PROVIDER_ADDR}")


## 0.1 Sanity check — agent is authorized and provider is paired

If either of these assertions fails, go back and run scenario 06 (agent
setup) and/or complete the master-provider pairing via the Provider App.


In [ ]:
assert terminal.is_agent_of(MASTER_ADDR, AGENT_ADDR), "Agent is not authorized for master — run scenario 06"
assert terminal.get_master_of(PROVIDER_ADDR).lower() == MASTER_ADDR.lower(), "Provider not paired to master"
print("Agent authorized and provider paired")


## 1. Client publishes a source


In [ ]:
source = publish_source(
    SourceInfo(
        name="agent-flow-demo",
        description="scenario 07 — agent-driven",
        logoUrl="https://example.com/logo.png",
        imageEnvs=ImageEnvironments(
            cpu="https://cipfs.ogpuscan.io/ipfs/QmNWFLL13ujf3KUTJvfNx42bA5fWDV96qqUdjY6nwpuwD9",
        ),
        minPayment=Web3.to_wei(0.01, "ether"),
        minAvailableLockup=0,
        maxExpiryDuration=86400,
        deliveryMethod=DeliveryMethod.MANUAL_CONFIRMATION,
    ),
    private_key=CLIENT_KEY,
)
print(f"Source: {source.address}")


## 2. Client publishes a task against the source


In [ ]:
task = publish_task(
    TaskInfo(
        source=source.address,
        config=TaskInput(function_name="demo_fn", data={"prompt": "agent-driven"}),
        expiryTime=int(time.time()) + 3600,
        payment=Web3.to_wei(0.01, "ether"),
    ),
    private_key=CLIENT_KEY,
)
print(f"Task: {task.address}  status={task.get_status()}")


## 3. Agent registers the provider to the source

The agent signs with its own key, but the on-chain effect is as if the master
itself called `Nexus.register(source, provider, env)`. The protocol checks
`isAgentOf(master, msg.sender)` to authorize the call.


In [ ]:
receipt = nexus.register(
    source=source.address,
    provider=PROVIDER_ADDR,
    env=Environment.CPU.value,
    signer=AGENT_KEY,  # ← agent signs, not master
)
print(f"register tx: {receipt.tx_hash}")

# Verify on-chain
print(f"Registrant id: {source.get_registrant_id(PROVIDER_ADDR)}")


## 4. Agent submits an attempt on the provider's behalf

Same pattern — agent signs, protocol authorizes via `isAgentOf`.


In [ ]:
receipt = nexus.attempt(
    task=task.address,
    provider=PROVIDER_ADDR,
    suggested_payment=Web3.to_wei(0.01, "ether"),
    signer=AGENT_KEY,
)
print(f"attempt tx: {receipt.tx_hash}")
print(f"Task status: {task.get_status()}")


## Takeaway

The agent key drove the on-chain scheduling for a task on behalf of the
master:

- registered the provider to a fresh source
- submitted an attempt against the task

The master's own key never signed anything — the agent delegation on
`Terminal.setAgent` (scenario 06) is the only authorization required. This is
the pattern that external schedulers like The Order use to operate at scale
without requiring the master to be online for every task.

What this notebook deliberately does **not** cover: the response submission
step. `Nexus.submitResponse` is not exposed in the SDK at any layer
(`ogpu.client`, `ogpu.protocol`, `ogpu.agent`) — it must come from the docker
source runtime that actually produced the compute output. Otherwise an agent
or any random script with a provider key could fabricate responses without
doing real work. Once a real provider runtime submits a response, the rest of
the lifecycle (`Response.confirm`, `Task FINALIZED`) is covered by the
client-side scenarios.
